# External Memory for AI Applications

This notebook builds a small AI application that uses MongoDB as external memory for a Qwen LLM orchestrated with LangGraph.

It demonstrates:

- Environment setup and model download
- `.env` configuration
- LLM call with basic context and streaming text-token output
- Saving conversation logs to MongoDB as history data
- Extracting durable user personal information into a separate MongoDB collection
- LangGraph tool calling with MongoDB-backed memory tools
- Turn-by-turn display of LLM tool calls and tool responses
- Summarizing the final context from system prompt, user message, tools, and history

Stack:

- LLM: `Qwen/Qwen3-4B-Instruct-2507`
- Agent framework: `langgraph`
- External memory: `mongodb`
- LLM adapter: OpenAI-compatible local endpoint, for example `vLLM`, `SGLang`, `LM Studio`, or another server that exposes `/v1/chat/completions`

References used while preparing this notebook:

- Qwen model card: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507
- LangGraph quickstart: https://docs.langchain.com/oss/python/langgraph/quickstart
- PyMongo docs: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/
- vLLM tool calling docs: https://docs.vllm.ai/en/latest/features/tool_calling/


## 1. Install Libraries

Run this cell once in a fresh notebook kernel. Restart the kernel after installation if imports do not resolve immediately.


In [ ]:
%pip install -U langgraph langchain langchain-core langchain-openai pymongo python-dotenv pydantic pandas huggingface_hub "transformers>=4.51.0" accelerate torch requests certifi


## 2. Start MongoDB

You can use MongoDB Atlas or a local MongoDB server. For a local Docker server, run this in a terminal:

```bash
docker run --name mongodb-memory -p 27017:27017 -d mongo:7
```

If the container already exists, start it with:

```bash
docker start mongodb-memory
```


## 3. Download or Serve the Qwen Model

For LangGraph tool calling, the smoothest path is an OpenAI-compatible local server. The Qwen model card recommends current inference stacks such as `vLLM` or `SGLang` for local serving.

Download the model weights with the next cell. This can take time and disk space.


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_LOCAL_DIR = Path("models/Qwen3-4B-Instruct-2507")

# Set local_dir=None if you prefer the normal Hugging Face cache.
model_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(MODEL_LOCAL_DIR),
)

print(f"Model downloaded to: {model_path}")


Start a local OpenAI-compatible model server in a separate terminal. Use a shorter context length if you hit GPU memory limits.

```bash
pip install -U vllm
vllm serve Qwen/Qwen3-4B-Instruct-2507 \
  --max-model-len 32768 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes
```

The server should expose `http://localhost:8000/v1`. Keep it running while executing the notebook. If your serving stack has native tool-call handling, use its equivalent settings and keep the same OpenAI-compatible `/v1` endpoint.


## 4. Create `.env`

This cell creates a local `.env` file only if one does not already exist. Update values if you use MongoDB Atlas, a different model server URL, or a different session/user id.


In [ ]:
from pathlib import Path

env_path = Path(".env")
env_template = """MONGODB_URI=mongodb://localhost:27017
MONGODB_DB=external_memory_demo
MONGODB_HISTORY_COLLECTION=chat_history
MONGODB_PROFILE_COLLECTION=user_personal_information
MONGODB_SUMMARY_COLLECTION=context_summaries
MONGODB_TOKEN_USAGE_COLLECTION=agent_token_usage

OPENAI_API_BASE=http://localhost:8000/v1
OPENAI_API_KEY=EMPTY
OPENAI_REQUEST_TIMEOUT=5
LLM_BACKEND=auto
QWEN_MODEL=Qwen/Qwen3-4B-Instruct-2507
QWEN_TEMPERATURE=0.2
QWEN_MAX_TOKENS=1024

SESSION_ID=demo-session-001
USER_ID=demo-user-001
MODEL_LOCAL_DIR=models/Qwen3-4B-Instruct-2507
"""

if env_path.exists():
    print(".env already exists. Keeping the existing file.")
else:
    env_path.write_text(env_template)
    print("Created .env. Review it before running production workloads.")


## 5. Load Configuration and Connect Clients

This section connects to MongoDB and configures Qwen.

`LLM_BACKEND=auto` avoids the common `llm.invoke(...)` connection error:

- If `OPENAI_API_BASE` is reachable, the notebook uses the OpenAI-compatible Qwen server.
- If that endpoint is not reachable and `MODEL_LOCAL_DIR` exists, the notebook falls back to local Transformers inference from the downloaded Qwen weights.
- Use `LLM_BACKEND=openai` only when you want the notebook to fail fast unless the model server is running.


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pymongo import ASCENDING, MongoClient

load_dotenv(override=True)

MONGODB_URI = os.getenv("MONGODB_URI", "mongodb://localhost:27017")
MONGODB_DB = os.getenv("MONGODB_DB", "external_memory_demo")
HISTORY_COLLECTION = os.getenv("MONGODB_HISTORY_COLLECTION", "chat_history")
PROFILE_COLLECTION = os.getenv("MONGODB_PROFILE_COLLECTION", "user_personal_information")
SUMMARY_COLLECTION = os.getenv("MONGODB_SUMMARY_COLLECTION", "context_summaries")

OPENAI_API_BASE = os.getenv("OPENAI_API_BASE", "http://localhost:8000/v1").rstrip("/")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "EMPTY")
OPENAI_REQUEST_TIMEOUT = float(os.getenv("OPENAI_REQUEST_TIMEOUT", "5"))
QWEN_MODEL = os.getenv("QWEN_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
QWEN_TEMPERATURE = float(os.getenv("QWEN_TEMPERATURE", "0.2"))
QWEN_MAX_TOKENS = int(os.getenv("QWEN_MAX_TOKENS", "1024"))
MODEL_LOCAL_DIR = Path(os.getenv("MODEL_LOCAL_DIR", "models/Qwen3-4B-Instruct-2507"))
LLM_BACKEND = os.getenv("LLM_BACKEND", "auto").strip().lower()

SESSION_ID = os.getenv("SESSION_ID", "demo-session-001")
USER_ID = os.getenv("USER_ID", "demo-user-001")

mongo_client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
mongo_client.admin.command("ping")

db = mongo_client[MONGODB_DB]
history_collection = db[HISTORY_COLLECTION]
profile_collection = db[PROFILE_COLLECTION]
summary_collection = db[SUMMARY_COLLECTION]

history_collection.create_index([("session_id", ASCENDING), ("created_at", ASCENDING)])
history_collection.create_index([("user_id", ASCENDING), ("created_at", ASCENDING)])
profile_collection.create_index("user_id", unique=True)
summary_collection.create_index([("session_id", ASCENDING), ("created_at", ASCENDING)])

print("MongoDB connected:", MONGODB_DB)
print("Requested LLM backend:", LLM_BACKEND)
print("Qwen endpoint:", OPENAI_API_BASE)
print("Local model directory:", MODEL_LOCAL_DIR)
print("Model:", QWEN_MODEL)
print()


In [ ]:
import copy
import json
import re
import uuid
from typing import Any

import requests
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage, ToolMessage


class LocalQwenChatModel:
    """Small LangChain-like chat wrapper around local Qwen weights.

    It is used only when the OpenAI-compatible endpoint is not reachable, so
    notebook cells can keep calling `llm.invoke(...)` without a localhost
    connection error. The model is loaded lazily on the first invocation.
    """

    supports_tool_calling = True

    def __init__(
        self,
        *,
        model_dir: Path,
        model_name: str,
        temperature: float = 0.2,
        max_new_tokens: int = 1024,
        bound_tools: list[Any] | None = None,
    ) -> None:
        self.model_dir = Path(model_dir)
        self.model_name = model_name
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens
        self.bound_tools = bound_tools or []
        self._tokenizer = None
        self._model = None

    def bind_tools(self, tools: list[Any]) -> "LocalQwenChatModel":
        clone = copy.copy(self)
        clone.bound_tools = list(tools)
        return clone

    def _load(self) -> None:
        if self._model is not None:
            return
        if not self.model_dir.exists():
            raise RuntimeError(
                f"Local model directory not found: {self.model_dir}. "
                "Run the model download cell or start the OpenAI-compatible Qwen server."
            )

        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        if torch.cuda.is_available():
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            device_map = "auto"
        else:
            dtype = torch.float32
            device_map = None

        self._tokenizer = AutoTokenizer.from_pretrained(
            self.model_dir,
            local_files_only=True,
            trust_remote_code=False,
        )
        self._model = AutoModelForCausalLM.from_pretrained(
            self.model_dir,
            torch_dtype=dtype,
            device_map=device_map,
            local_files_only=True,
            trust_remote_code=False,
        )
        self._model.eval()

    def _tool_instruction(self) -> str:
        if not self.bound_tools:
            return ""

        tool_lines = []
        for selected_tool in self.bound_tools:
            args_schema = getattr(selected_tool, "args_schema", None)
            if args_schema and hasattr(args_schema, "model_json_schema"):
                schema = args_schema.model_json_schema()
            else:
                schema = {}
            tool_lines.append(
                f"- {selected_tool.name}: {selected_tool.description}\n"
                f"  Args JSON schema: {json.dumps(schema, ensure_ascii=False)}"
            )

        return "\n".join(
            [
                "You have access to tools. When a tool is needed, return only valid JSON in this shape:",
                '{"tool_calls": [{"name": "tool_name", "args": {}}]}',
                "",
                "When you have enough information to answer, return normal assistant text instead of JSON.",
                "Available tools:",
                "\n".join(tool_lines),
            ]
        )

    def _coerce_messages(self, messages: Any) -> list[dict[str, str]]:
        if isinstance(messages, str):
            coerced = [{"role": "user", "content": messages}]
        else:
            coerced = []
            for message in messages:
                if isinstance(message, SystemMessage):
                    role = "system"
                    content = message.content
                elif isinstance(message, HumanMessage):
                    role = "user"
                    content = message.content
                elif isinstance(message, AIMessage):
                    role = "assistant"
                    content = message.content
                elif isinstance(message, ToolMessage):
                    role = "user"
                    content = f"Tool result from {getattr(message, 'name', 'tool')}: {message.content}"
                elif isinstance(message, BaseMessage):
                    role = "user"
                    content = message.content
                else:
                    role = "user"
                    content = str(message)
                coerced.append({"role": role, "content": str(content)})

        tool_instruction = self._tool_instruction()
        if tool_instruction:
            if coerced and coerced[0]["role"] == "system":
                coerced[0]["content"] = coerced[0]["content"] + "\n\n" + tool_instruction
            else:
                coerced.insert(0, {"role": "system", "content": tool_instruction})
        return coerced

    def count_message_tokens(self, messages: Any) -> int:
        """Count prompt tokens with the local Qwen tokenizer."""
        self._load()
        chat_messages = self._coerce_messages(messages)
        prompt = self._tokenizer.apply_chat_template(
            chat_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        return len(self._tokenizer(prompt, add_special_tokens=False)["input_ids"])

    def count_text_tokens(self, text: Any) -> int:
        """Count output tokens with the local Qwen tokenizer."""
        self._load()
        return len(self._tokenizer(str(text), add_special_tokens=False)["input_ids"])

    @staticmethod
    def _extract_json_object(text: str) -> dict[str, Any] | None:
        cleaned = text.strip()
        fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.DOTALL)
        if fenced:
            cleaned = fenced.group(1)
        else:
            first = cleaned.find("{")
            last = cleaned.rfind("}")
            if first >= 0 and last >= first:
                cleaned = cleaned[first : last + 1]
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return None

    def _maybe_parse_tool_calls(self, content: str) -> AIMessage | None:
        if not self.bound_tools:
            return None
        data = self._extract_json_object(content)
        if not data or "tool_calls" not in data:
            return None

        known_tool_names = {selected_tool.name for selected_tool in self.bound_tools}
        tool_calls = []
        for call in data.get("tool_calls", []):
            name = call.get("name")
            if name not in known_tool_names:
                continue
            args = call.get("args", {})
            if not isinstance(args, dict):
                args = {}
            tool_calls.append(
                {
                    "name": name,
                    "args": args,
                    "id": call.get("id") or f"call_{uuid.uuid4().hex[:12]}",
                }
            )

        if not tool_calls:
            return None
        return AIMessage(
            content="",
            tool_calls=tool_calls,
            response_metadata={"backend": "local-transformers", "model": self.model_name},
        )

    def invoke(self, messages: Any, **_: Any) -> AIMessage:
        self._load()
        chat_messages = self._coerce_messages(messages)
        prompt = self._tokenizer.apply_chat_template(
            chat_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        inputs = self._tokenizer([prompt], return_tensors="pt")
        first_device = next(self._model.parameters()).device
        inputs = {key: value.to(first_device) for key, value in inputs.items()}

        generation_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "do_sample": self.temperature > 0,
            "pad_token_id": self._tokenizer.eos_token_id,
        }
        if generation_kwargs["do_sample"]:
            generation_kwargs["temperature"] = self.temperature

        generated = self._model.generate(**inputs, **generation_kwargs)
        output_ids = generated[0][inputs["input_ids"].shape[-1] :]
        content = self._tokenizer.decode(output_ids, skip_special_tokens=True).strip()

        tool_call_message = self._maybe_parse_tool_calls(content)
        if tool_call_message is not None:
            return tool_call_message

        return AIMessage(
            content=content,
            response_metadata={"backend": "local-transformers", "model": self.model_name},
        )

    def stream(self, messages: Any, **_: Any):
        """Yield text chunks so the local fallback behaves like ChatOpenAI.stream(...)."""
        self._load()
        from threading import Thread

        from transformers import TextIteratorStreamer

        chat_messages = self._coerce_messages(messages)
        prompt = self._tokenizer.apply_chat_template(
            chat_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        inputs = self._tokenizer([prompt], return_tensors="pt")
        first_device = next(self._model.parameters()).device
        inputs = {key: value.to(first_device) for key, value in inputs.items()}

        generation_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "do_sample": self.temperature > 0,
            "pad_token_id": self._tokenizer.eos_token_id,
        }
        if generation_kwargs["do_sample"]:
            generation_kwargs["temperature"] = self.temperature

        streamer = TextIteratorStreamer(
            self._tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )
        thread = Thread(
            target=self._model.generate,
            kwargs={**inputs, **generation_kwargs, "streamer": streamer},
        )
        thread.start()

        for text in streamer:
            if text:
                yield AIMessageChunk(
                    content=text,
                    response_metadata={"backend": "local-transformers", "model": self.model_name},
                )
        thread.join()


def openai_models_url(base_url: str) -> str:
    return f"{base_url.rstrip('/')}/models"


def openai_headers() -> dict[str, str]:
    if OPENAI_API_KEY and OPENAI_API_KEY != "EMPTY":
        return {"Authorization": f"Bearer {OPENAI_API_KEY}"}
    return {}


def check_openai_endpoint() -> tuple[bool, str]:
    try:
        response = requests.get(
            openai_models_url(OPENAI_API_BASE),
            headers=openai_headers(),
            timeout=OPENAI_REQUEST_TIMEOUT,
        )
        if response.ok:
            return True, f"{response.status_code} OK"
        return False, f"HTTP {response.status_code}: {response.text[:300]}"
    except requests.RequestException as exc:
        return False, str(exc)


def create_openai_llm() -> ChatOpenAI:
    kwargs = {
        "model": QWEN_MODEL,
        "base_url": OPENAI_API_BASE,
        "api_key": OPENAI_API_KEY,
        "temperature": QWEN_TEMPERATURE,
        "max_tokens": QWEN_MAX_TOKENS,
        "timeout": OPENAI_REQUEST_TIMEOUT,
        "max_retries": 0,
    }
    try:
        return ChatOpenAI(**kwargs, stream_usage=True)
    except TypeError:
        return ChatOpenAI(**kwargs)


def create_local_llm() -> LocalQwenChatModel:
    return LocalQwenChatModel(
        model_dir=MODEL_LOCAL_DIR,
        model_name=QWEN_MODEL,
        temperature=QWEN_TEMPERATURE,
        max_new_tokens=QWEN_MAX_TOKENS,
    )


def build_llm() -> tuple[Any, str, str]:
    if LLM_BACKEND not in {"auto", "openai", "transformers", "local"}:
        raise ValueError("LLM_BACKEND must be one of: auto, openai, transformers, local")

    endpoint_ok, endpoint_detail = check_openai_endpoint()
    if LLM_BACKEND in {"auto", "openai"} and endpoint_ok:
        return create_openai_llm(), "openai-compatible", f"Endpoint ready: {endpoint_detail}"

    if LLM_BACKEND == "openai":
        raise RuntimeError(
            "OPENAI-compatible Qwen endpoint is not reachable. "
            f"Checked {openai_models_url(OPENAI_API_BASE)}. Detail: {endpoint_detail}"
        )

    if LLM_BACKEND in {"auto", "transformers", "local"} and MODEL_LOCAL_DIR.exists():
        return (
            create_local_llm(),
            "local-transformers",
            "OpenAI-compatible endpoint not reachable; using local Qwen weights. "
            f"Endpoint detail: {endpoint_detail}",
        )

    raise RuntimeError(
        "No usable Qwen backend found. Start vLLM/SGLang at OPENAI_API_BASE or run the model download cell. "
        f"Endpoint detail: {endpoint_detail}. Local model dir exists: {MODEL_LOCAL_DIR.exists()}"
    )


llm, LLM_BACKEND_ACTIVE, LLM_HEALTH = build_llm()
print("Active LLM backend:", LLM_BACKEND_ACTIVE)
print("LLM health:", LLM_HEALTH)

try:
    print("Sending test request with llm.invoke(...)")
    test_response = llm.invoke("Say exactly: Connection Successful!")
except Exception as exc:
    if LLM_BACKEND == "auto" and LLM_BACKEND_ACTIVE == "openai-compatible" and MODEL_LOCAL_DIR.exists():
        print("OpenAI-compatible invoke failed; switching to local Transformers Qwen.")
        print("Original error:", exc)
        llm = create_local_llm()
        LLM_BACKEND_ACTIVE = "local-transformers"
        LLM_HEALTH = "Fallback after OpenAI-compatible invoke failure."
        test_response = llm.invoke("Say exactly: Connection Successful!")
    else:
        raise

print("\n--- Success! ---")
print(test_response.content)


## Optional Proxy Note

The notebook no longer sets `openai.proxy` manually. Modern `langchain-openai` uses the OpenAI Python client underneath, and local `localhost` calls usually should not go through an HTTP proxy.

If your environment requires a proxy for a remote OpenAI-compatible endpoint, set it outside the notebook with standard environment variables such as `HTTP_PROXY` and `HTTPS_PROXY`, then restart the kernel.


In [ ]:
# Optional endpoint inspection. This runs only when the active backend is the OpenAI-compatible server.
if LLM_BACKEND_ACTIVE == "openai-compatible":
    response = requests.get(
        openai_models_url(OPENAI_API_BASE),
        headers=openai_headers(),
        timeout=OPENAI_REQUEST_TIMEOUT,
    )
    print(response.status_code)
    print(response.text[:1000])
else:
    print(f"Skipping /models check because active backend is {LLM_BACKEND_ACTIVE}.")


## 6. Shared Helpers for MongoDB Memory

These helpers convert LangChain messages into MongoDB documents, reload recent history, and render memory as text for prompts.


In [ ]:
import json
import re
import uuid
from typing import Any

import pandas as pd
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def message_role(message: BaseMessage) -> str:
    return getattr(message, "type", message.__class__.__name__.replace("Message", "").lower())


def serialize_message(
    message: BaseMessage,
    *,
    session_id: str,
    user_id: str,
    turn_id: str,
    source: str,
) -> dict[str, Any]:
    doc = {
        "session_id": session_id,
        "user_id": user_id,
        "turn_id": turn_id,
        "source": source,
        "role": message_role(message),
        "content": message.content,
        "created_at": utc_now(),
        "additional_kwargs": getattr(message, "additional_kwargs", {}),
        "response_metadata": getattr(message, "response_metadata", {}),
    }

    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        doc["tool_calls"] = tool_calls

    tool_call_id = getattr(message, "tool_call_id", None)
    if tool_call_id:
        doc["tool_call_id"] = tool_call_id

    return doc


def save_messages(
    messages: list[BaseMessage],
    *,
    session_id: str = SESSION_ID,
    user_id: str = USER_ID,
    turn_id: str | None = None,
    source: str = "notebook",
) -> str:
    turn_id = turn_id or str(uuid.uuid4())
    docs = [
        serialize_message(
            message,
            session_id=session_id,
            user_id=user_id,
            turn_id=turn_id,
            source=source,
        )
        for message in messages
    ]
    if docs:
        history_collection.insert_many(docs)
    return turn_id


def load_recent_history(session_id: str = SESSION_ID, limit: int = 12) -> list[dict[str, Any]]:
    docs = list(
        history_collection.find({"session_id": session_id}, {"_id": 0})
        .sort("created_at", -1)
        .limit(limit)
    )
    return list(reversed(docs))


def render_history(history_docs: list[dict[str, Any]]) -> str:
    if not history_docs:
        return "No prior history."
    lines = []
    for doc in history_docs:
        role = doc.get("role", "unknown")
        content = doc.get("content", "")
        if isinstance(content, list):
            content = json.dumps(content, ensure_ascii=False)
        content = str(content).strip().replace("\n", " ")
        lines.append(f"{role}: {content[:600]}")
    return "\n".join(lines)


def extract_json_object(text: str) -> dict[str, Any]:
    cleaned = text.strip()
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.DOTALL)
    if fenced:
        cleaned = fenced.group(1)
    else:
        first = cleaned.find("{")
        last = cleaned.rfind("}")
        if first >= 0 and last >= first:
            cleaned = cleaned[first : last + 1]
    return json.loads(cleaned)


## 7. LLM with Basic Context and Streaming Tokens

This is the simplest pattern: pass a system prompt, a small context block, and the user's message to the LLM. The cell prints streaming output chunks as they arrive, then stores the completed text in `basic_response` for later reuse.


In [ ]:
SYSTEM_PROMPT = """
You are MemoryGuide, an assistant that explains external memory patterns for AI applications.
Be concise, practical, and explicit about what is stored in short-term context versus external memory.
""".strip()

basic_context = """
The application uses MongoDB as external memory.
Conversation history is stored in the chat_history collection.
Durable user profile facts are stored in the user_personal_information collection.
""".strip()

user_question = "Explain why an AI app should store conversation history outside the LLM context window."

basic_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=f"Context:\n{basic_context}\n\nUser question:\n{user_question}"),
]


def chunk_to_text(chunk: Any) -> str:
    content = getattr(chunk, "content", chunk)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text", item.get("content", ""))))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content) if content is not None else ""


def stream_llm_text(messages: list[BaseMessage] | str) -> str:
    """Print LLM output token/chunk text as it arrives and return the full text."""
    print("Streaming LLM text chunks:")
    print("-" * 72)
    chunks = []
    for chunk in llm.stream(messages):
        text = chunk_to_text(chunk)
        if text:
            print(text, end="", flush=True)
            chunks.append(text)
    print("\n" + "-" * 72)
    return "".join(chunks)


basic_response_text = stream_llm_text(basic_messages)
basic_response = AIMessage(
    content=basic_response_text,
    response_metadata={"backend": LLM_BACKEND_ACTIVE, "streamed": True},
)


In [ ]:
# Backend-aware model inspection. This avoids hard-coded localhost ports.
if LLM_BACKEND_ACTIVE == "openai-compatible":
    response = requests.get(
        openai_models_url(OPENAI_API_BASE),
        headers=openai_headers(),
        timeout=OPENAI_REQUEST_TIMEOUT,
    )
    print(response.status_code)
    print(response.text[:1000])
else:
    print(f"Active backend is {LLM_BACKEND_ACTIVE}; no HTTP model endpoint is required.")


## 8. Save LLM Logs to MongoDB as History Data

This pattern stores each user and assistant message as an append-only history log. History logs are useful for debugging, analytics, replay, and future context retrieval.


In [ ]:
turn_id = str(uuid.uuid4())

user_message = HumanMessage(
    content=(
        "Hi, my name is Mina. I am a product manager in Bangkok. "
        "I prefer concise answers and I am building an AI memory demo with MongoDB."
    )
)

user_response_text = stream_llm_text(user_message)
assistant_message = AIMessage(
    content=user_response_text,
    response_metadata={"backend": LLM_BACKEND_ACTIVE, "streamed": True},
)
save_messages(
    [user_message, assistant_message],
    turn_id=turn_id,
    source="basic_chat_log",
)
print("Saved turn:", turn_id)


In [ ]:
history_preview = load_recent_history(limit=6)
pd.DataFrame(history_preview)[["created_at", "role", "source", "content"]]


## 9. Extract Durable User Personal Information

History data is a log. User personal information is a structured memory record. This section asks the LLM to extract durable facts and saves them to a separate MongoDB collection.


In [ ]:
from pydantic import BaseModel, Field


class UserPersonalInfo(BaseModel):
    name: str | None = Field(default=None, description="The user's preferred name")
    location: str | None = Field(default=None, description="The user's city, region, or country")
    occupation: str | None = Field(default=None, description="The user's job or role")
    preferences: list[str] = Field(default_factory=list, description="Stable communication or product preferences")
    projects: list[str] = Field(default_factory=list, description="Durable projects or goals the user is working on")
    sensitive_notes: list[str] = Field(
        default_factory=list,
        description="Potentially sensitive facts. Store only when explicitly needed and allowed.",
    )


def extract_user_personal_info(message: str) -> UserPersonalInfo:
    schema = UserPersonalInfo.model_json_schema()
    extraction_prompt = f"""
Extract durable user personal information from the message.
Return only valid JSON matching this schema:
{json.dumps(schema, indent=2)}

Rules:
- Extract stable facts that could help future conversations.
- Do not invent facts.
- Keep sensitive_notes empty unless the user clearly provided sensitive information and storage is justified.

Message:
{message}
""".strip()

    response = stream_llm_text([
        SystemMessage(content="You extract structured memory as strict JSON."),
        HumanMessage(content=extraction_prompt),
    ])
    data = extract_json_object(response)
    return UserPersonalInfo.model_validate(data)


def upsert_user_profile(user_id: str, info: UserPersonalInfo, source_message: str) -> dict[str, Any]:
    extracted_data = info.model_dump()
    durable_fields = {
        key: value
        for key, value in extracted_data.items()
        if value not in (None, "", [], {})
    }
    profile_updates = {f"profile.{key}": value for key, value in durable_fields.items()}

    profile_collection.update_one(
        {"user_id": user_id},
        {
            "$set": {
                **profile_updates,
                "user_id": user_id,
                "updated_at": utc_now(),
            },
            "$setOnInsert": {"created_at": utc_now()},
            "$push": {
                "extraction_events": {
                    "source_message": source_message,
                    "extracted": extracted_data,
                    "saved_fields": sorted(durable_fields.keys()),
                    "created_at": utc_now(),
                }
            },
        },
        upsert=True,
    )
    return profile_collection.find_one({"user_id": user_id}, {"_id": 0})


In [ ]:
profile_info = extract_user_personal_info(user_message.content)
profile_doc = upsert_user_profile(USER_ID, profile_info, user_message.content)

pprint(profile_doc)


## 10. LLM with LangGraph Tool Calling and Turn Trace

Now the LLM can call tools that read and write MongoDB. The graph loops through `llm -> tools -> llm` until the assistant returns a final answer. The run cell below streams graph updates and prints each turn: user message, LLM tool call, tool response, final LLM response, and MongoDB log save.

The demo agent has tools for profile lookup/save, chat-history search, current datetime, memory statistics, recent profile memory events, saving working notes, and listing working notes. The system prompt asks the model to call one tool per assistant turn so the loop is visible in the output.

The run also prints token usage for every LLM turn and a final total across the full agent loop. Provider-reported usage is used when available; otherwise the notebook estimates usage from the local tokenizer or a rough text estimate.

Visible stream rendering prints each received model chunk in small flushed slices, so tool-call JSON and metadata are visibly streamed even when the backend sends a large chunk.


In [ ]:
import operator
from typing import Literal

from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict

agent_notes_collection = db["agent_notes"]
agent_notes_collection.create_index([("user_id", ASCENDING), ("created_at", ASCENDING)])
agent_notes_collection.create_index([("session_id", ASCENDING), ("created_at", ASCENDING)])


def json_safe(value: Any) -> Any:
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    return value


@tool
def lookup_user_profile(user_id: str) -> dict[str, Any]:
    """Look up durable user profile memory in MongoDB by user_id."""
    doc = profile_collection.find_one({"user_id": user_id}, {"_id": 0})
    if not doc:
        return {"found": False, "user_id": user_id, "profile": {}}
    return json_safe(doc)


@tool
def save_user_personal_information(user_id: str, facts: dict[str, Any], source_message: str = "") -> dict[str, Any]:
    """Save durable user personal information to MongoDB. Use this for stable preferences, identity facts, projects, goals, and role information."""
    update_fields = {
        f"profile.{key}": value
        for key, value in facts.items()
        if value not in (None, "", [], {})
    }
    if not update_fields:
        return {"saved": False, "reason": "No non-empty facts provided."}

    profile_collection.update_one(
        {"user_id": user_id},
        {
            "$set": {**update_fields, "user_id": user_id, "updated_at": utc_now()},
            "$setOnInsert": {"created_at": utc_now()},
            "$push": {
                "tool_events": {
                    "tool": "save_user_personal_information",
                    "facts": facts,
                    "source_message": source_message,
                    "created_at": utc_now(),
                }
            },
        },
        upsert=True,
    )
    return {"saved": True, "fields": sorted(update_fields.keys())}


@tool
def search_chat_history(session_id: str, query: str = "", limit: int = 5) -> list[dict[str, Any]]:
    """Search recent chat history in MongoDB. Query is matched as a case-insensitive substring against message content."""
    mongo_filter: dict[str, Any] = {"session_id": session_id}
    if query:
        mongo_filter["content"] = {"$regex": re.escape(query), "$options": "i"}

    docs = list(
        history_collection.find(mongo_filter, {"_id": 0})
        .sort("created_at", -1)
        .limit(max(1, min(limit, 20)))
    )
    return json_safe(docs)


@tool
def get_current_datetime(timezone_name: str = "Asia/Bangkok") -> dict[str, Any]:
    """Return the current date and time for a timezone. Use this when an answer needs a timestamp or local date context."""
    from zoneinfo import ZoneInfo

    now = datetime.now(ZoneInfo(timezone_name))
    return {
        "timezone": timezone_name,
        "iso_datetime": now.isoformat(),
        "date": now.date().isoformat(),
        "time": now.strftime("%H:%M:%S"),
    }


@tool
def get_memory_stats(user_id: str, session_id: str) -> dict[str, Any]:
    """Count memory records in MongoDB for a user and session. Useful for showing what the external memory currently contains."""
    return {
        "user_id": user_id,
        "session_id": session_id,
        "history_messages": history_collection.count_documents({"session_id": session_id}),
        "user_profile_exists": profile_collection.count_documents({"user_id": user_id}) > 0,
        "context_summaries": summary_collection.count_documents({"session_id": session_id}),
        "agent_notes": agent_notes_collection.count_documents({"user_id": user_id, "session_id": session_id}),
    }


@tool
def list_recent_user_memory_events(user_id: str, limit: int = 5) -> list[dict[str, Any]]:
    """List recent profile extraction and profile tool-save events for a user."""
    profile_doc = profile_collection.find_one({"user_id": user_id}, {"_id": 0}) or {}
    events = []
    for event in profile_doc.get("extraction_events", []):
        events.append({"event_type": "profile_extraction", **event})
    for event in profile_doc.get("tool_events", []):
        events.append({"event_type": "profile_tool_save", **event})
    events.sort(key=lambda item: item.get("created_at", datetime.min.replace(tzinfo=timezone.utc)), reverse=True)
    return json_safe(events[: max(1, min(limit, 20))])


@tool
def save_agent_note(user_id: str, session_id: str, title: str, note: str, tags: list[str] | None = None) -> dict[str, Any]:
    """Save a non-sensitive working note to the agent_notes MongoDB collection for later retrieval."""
    doc = {
        "user_id": user_id,
        "session_id": session_id,
        "title": title,
        "note": note,
        "tags": tags or [],
        "created_at": utc_now(),
    }
    result = agent_notes_collection.insert_one(doc)
    return {"saved": True, "note_id": str(result.inserted_id), "title": title, "tags": tags or []}


@tool
def list_recent_agent_notes(user_id: str, session_id: str, limit: int = 5) -> list[dict[str, Any]]:
    """List recent non-sensitive working notes saved by the agent for a user and session."""
    docs = list(
        agent_notes_collection.find({"user_id": user_id, "session_id": session_id}, {"_id": 0})
        .sort("created_at", -1)
        .limit(max(1, min(limit, 20)))
    )
    return json_safe(docs)


tools = [
    lookup_user_profile,
    save_user_personal_information,
    search_chat_history,
    get_current_datetime,
    get_memory_stats,
    list_recent_user_memory_events,
    save_agent_note,
    list_recent_agent_notes,
]
tools_by_name = {memory_tool.name: memory_tool for memory_tool in tools}
llm_with_tools = llm.bind_tools(tools)


class MemoryAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    session_id: str
    user_id: str
    llm_calls: int


In [ ]:
TOOL_AGENT_SYSTEM_PROMPT = """
You are MemoryGuide, an assistant with external memory in MongoDB.

Memory policy:
- Use lookup_user_profile when a question depends on durable personal information.
- Use search_chat_history when prior conversation details may help.
- Use save_user_personal_information only for stable facts the user explicitly shares, such as name, role, location, preferences, projects, or goals.
- Use get_current_datetime when the user asks for a dated note, report, or current local context.
- Use get_memory_stats to inspect how much external memory exists for this user/session.
- Use list_recent_user_memory_events and list_recent_agent_notes when auditing or explaining memory state.
- Use save_agent_note for non-sensitive working notes that are useful but not durable user profile facts.
- Do not store secrets, credentials, payment data, health data, or other sensitive data unless the user explicitly asks and the application policy allows it.
- When you use memory, briefly explain which memory helped.
- For this notebook demo, call at most one tool per assistant turn so the agent loop is visible.

Runtime identifiers:
- user_id: {user_id}
- session_id: {session_id}

Recent MongoDB history:
{history}
""".strip()

try:
    from langchain_core.messages import message_chunk_to_message
except ImportError:
    message_chunk_to_message = None


TRACE_STREAM_CALL_MODEL = True
LLM_TOKEN_USAGE_LOG: list[dict[str, Any]] = []
STREAM_DISPLAY_CHUNK_SIZE = 8
STREAM_DISPLAY_DELAY_SECONDS = 0.015


def normalize_token_usage(raw_usage: dict[str, Any] | None) -> dict[str, int] | None:
    if not raw_usage:
        return None

    prompt_tokens = raw_usage.get("prompt_tokens", raw_usage.get("input_tokens"))
    completion_tokens = raw_usage.get("completion_tokens", raw_usage.get("output_tokens"))
    total_tokens = raw_usage.get("total_tokens", raw_usage.get("total"))

    if total_tokens is None and prompt_tokens is not None and completion_tokens is not None:
        total_tokens = prompt_tokens + completion_tokens

    usage = {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
    }
    if all(value is None for value in usage.values()):
        return None
    return {key: int(value) if value is not None else None for key, value in usage.items()}


def extract_provider_token_usage(*objects: Any) -> dict[str, int] | None:
    for obj in objects:
        if obj is None:
            continue

        candidates = []
        usage_metadata = getattr(obj, "usage_metadata", None)
        if usage_metadata:
            candidates.append(usage_metadata)

        response_metadata = getattr(obj, "response_metadata", None) or {}
        if response_metadata:
            candidates.extend([
                response_metadata.get("token_usage"),
                response_metadata.get("usage"),
                response_metadata,
            ])

        for candidate in candidates:
            usage = normalize_token_usage(candidate)
            if usage:
                return usage
    return None


def rough_token_estimate(text: Any) -> int:
    text = str(text or "")
    if not text:
        return 0
    return max(1, len(text) // 4)


def rendered_messages_for_estimate(messages: list[BaseMessage]) -> str:
    lines = []
    for message in messages:
        role = message_role(message)
        content = getattr(message, "content", "")
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            content = f"{content}\nTool calls: {json.dumps(tool_calls, ensure_ascii=False, default=str)}"
        lines.append(f"{role}: {content}")
    return "\n".join(lines)


def estimate_prompt_tokens(messages: list[BaseMessage]) -> int:
    if hasattr(llm_with_tools, "count_message_tokens"):
        try:
            return llm_with_tools.count_message_tokens(messages)
        except Exception:
            pass
    return rough_token_estimate(rendered_messages_for_estimate(messages))


def output_text_for_usage(response: AIMessage, streamed_text: str) -> str:
    if streamed_text.strip():
        return streamed_text

    content = trace_chunk_to_text(response)
    if content.strip():
        return content

    tool_calls = getattr(response, "tool_calls", None) or []
    if tool_calls:
        return json.dumps(tool_calls, ensure_ascii=False, default=str)
    return ""


def estimate_completion_tokens(response: AIMessage, streamed_text: str) -> int:
    output_text = output_text_for_usage(response, streamed_text)
    if hasattr(llm_with_tools, "count_text_tokens"):
        try:
            return llm_with_tools.count_text_tokens(output_text)
        except Exception:
            pass
    return rough_token_estimate(output_text)


def token_usage_for_turn(
    *,
    turn_number: int,
    messages: list[BaseMessage],
    response: AIMessage,
    response_chunk: Any,
    streamed_text: str,
) -> dict[str, Any]:
    provider_usage = extract_provider_token_usage(response, response_chunk)
    if provider_usage:
        usage = {**provider_usage, "source": "provider"}
    else:
        prompt_tokens = estimate_prompt_tokens(messages)
        completion_tokens = estimate_completion_tokens(response, streamed_text)
        usage = {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
            "source": "estimated",
        }

    usage.update({
        "turn": turn_number,
        "llm_call": len(LLM_TOKEN_USAGE_LOG) + 1,
        "backend": LLM_BACKEND_ACTIVE,
    })
    return usage


def format_token_value(value: Any) -> str:
    return "unknown" if value is None else f"{value:,}"


def print_turn_token_usage(usage: dict[str, Any]) -> None:
    print("Token usage:")
    print(f"- Prompt tokens: {format_token_value(usage.get('prompt_tokens'))}")
    print(f"- Completion tokens: {format_token_value(usage.get('completion_tokens'))}")
    print(f"- Total tokens: {format_token_value(usage.get('total_tokens'))}")
    print(f"- Source: {usage.get('source')} ({usage.get('backend')})")


def summarize_token_usage(usage_log: list[dict[str, Any]]) -> dict[str, Any]:
    totals = {
        "llm_calls": len(usage_log),
        "prompt_tokens": sum(row.get("prompt_tokens") or 0 for row in usage_log),
        "completion_tokens": sum(row.get("completion_tokens") or 0 for row in usage_log),
        "total_tokens": sum(row.get("total_tokens") or 0 for row in usage_log),
    }
    return totals


def print_stream_piece(text: str, *, chunk_size: int = STREAM_DISPLAY_CHUNK_SIZE, delay: float = STREAM_DISPLAY_DELAY_SECONDS) -> None:
    """Print stream payload in small flushed slices so notebook output visibly updates."""
    import time

    if not text:
        return
    for start in range(0, len(text), chunk_size):
        print(text[start : start + chunk_size], end="", flush=True)
        if delay:
            time.sleep(delay)


def trace_chunk_to_text(chunk: Any) -> str:
    content = getattr(chunk, "content", chunk)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text", item.get("content", ""))))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content) if content is not None else ""


def trace_tool_call_delta_text(chunk: Any) -> str:
    """Render streamed tool-call deltas, which often arrive outside normal content."""
    tool_call_chunks = getattr(chunk, "tool_call_chunks", None) or []
    if not tool_call_chunks:
        return ""
    rendered = []
    for tool_chunk in tool_call_chunks:
        name = tool_chunk.get("name") if isinstance(tool_chunk, dict) else getattr(tool_chunk, "name", None)
        args = tool_chunk.get("args") if isinstance(tool_chunk, dict) else getattr(tool_chunk, "args", None)
        call_id = tool_chunk.get("id") if isinstance(tool_chunk, dict) else getattr(tool_chunk, "id", None)
        delta = {key: value for key, value in {"id": call_id, "name": name, "args": args}.items() if value not in (None, "")}
        if delta:
            rendered.append(json.dumps(delta, ensure_ascii=False, default=str))
    return "".join(rendered)


def trace_short_json(value: Any, limit: int = 1200) -> str:
    text = json.dumps(json_safe(value), ensure_ascii=False, indent=2, default=str)
    if len(text) > limit:
        return text[:limit] + "\n...<truncated>"
    return text


def coerce_streamed_ai_message(response_chunk: Any, streamed_text: str) -> AIMessage:
    if response_chunk is None:
        return AIMessage(content=streamed_text)

    response = response_chunk
    if message_chunk_to_message is not None:
        try:
            response = message_chunk_to_message(response_chunk)
        except Exception:
            response = response_chunk

    parsed_tool_call = None
    if not getattr(response, "tool_calls", None) and hasattr(llm_with_tools, "_maybe_parse_tool_calls"):
        parsed_tool_call = llm_with_tools._maybe_parse_tool_calls(getattr(response, "content", streamed_text))
    if parsed_tool_call is not None:
        return parsed_tool_call

    if isinstance(response, AIMessage):
        return response

    return AIMessage(
        content=getattr(response, "content", streamed_text),
        tool_calls=getattr(response, "tool_calls", []) or [],
        response_metadata=getattr(response, "response_metadata", {}),
    )


def stream_call_model_response(messages: list[BaseMessage], *, turn_number: int) -> AIMessage:
    print(f"\nTurn {turn_number} | call_model")
    print("=" * 72)
    print("LLM stream output:")

    aggregate_chunk = None
    streamed_parts = []
    saw_text = False
    saw_tool_delta = False

    for chunk in llm_with_tools.stream(messages):
        if aggregate_chunk is None:
            aggregate_chunk = chunk
        else:
            try:
                aggregate_chunk = aggregate_chunk + chunk
            except Exception:
                pass

        text = trace_chunk_to_text(chunk)
        if text:
            print_stream_piece(text)
            streamed_parts.append(text)
            saw_text = True

        tool_delta_text = trace_tool_call_delta_text(chunk)
        if tool_delta_text:
            print_stream_piece(tool_delta_text)
            saw_tool_delta = True

    if saw_text or saw_tool_delta:
        print()
    if saw_tool_delta and not saw_text:
        print("[tool-call metadata stream complete]")
    elif not saw_text:
        print("[no streamed text chunks]")

    streamed_text = "".join(streamed_parts)
    response = coerce_streamed_ai_message(aggregate_chunk, streamed_text)
    tool_calls = getattr(response, "tool_calls", None) or []
    if tool_calls:
        for index, tool_call in enumerate(tool_calls, start=1):
            print(f"LLM call tool #{index}: {tool_call['name']}")
            print("Arguments:")
            print(trace_short_json(tool_call.get("args", {})))
    else:
        print("LLM final text response complete.")

    usage = token_usage_for_turn(
        turn_number=turn_number,
        messages=messages,
        response=response,
        response_chunk=aggregate_chunk,
        streamed_text=streamed_text,
    )
    LLM_TOKEN_USAGE_LOG.append(usage)
    print_turn_token_usage(usage)

    return response


def build_tool_agent_messages(state: MemoryAgentState) -> list[BaseMessage]:
    history_text = render_history(load_recent_history(state["session_id"], limit=8))
    system_message = SystemMessage(
        content=TOOL_AGENT_SYSTEM_PROMPT.format(
            user_id=state["user_id"],
            session_id=state["session_id"],
            history=history_text,
        )
    )
    return [system_message] + state["messages"]


def call_model(state: MemoryAgentState) -> dict[str, Any]:
    messages = build_tool_agent_messages(state)
    turn_number = 1 + (state.get("llm_calls", 0) * 2)
    if TRACE_STREAM_CALL_MODEL:
        response = stream_call_model_response(messages, turn_number=turn_number)
    else:
        response = llm_with_tools.invoke(messages)
        streamed_text = trace_chunk_to_text(response)
        usage = token_usage_for_turn(
            turn_number=turn_number,
            messages=messages,
            response=response,
            response_chunk=response,
            streamed_text=streamed_text,
        )
        LLM_TOKEN_USAGE_LOG.append(usage)
        print_turn_token_usage(usage)
    return {
        "messages": [response],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }


def tool_node(state: MemoryAgentState) -> dict[str, Any]:
    last_message = state["messages"][-1]
    results = []

    for tool_call in last_message.tool_calls:
        selected_tool = tools_by_name[tool_call["name"]]
        observation = selected_tool.invoke(tool_call["args"])
        if not isinstance(observation, str):
            observation = json.dumps(json_safe(observation), ensure_ascii=False, default=str)
        results.append(
            ToolMessage(
                content=observation,
                tool_call_id=tool_call["id"],
                name=tool_call["name"],
            )
        )

    return {"messages": results}


def should_continue(state: MemoryAgentState) -> Literal["tool_node", "save_agent_log"]:
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tool_node"
    return "save_agent_log"


def save_agent_log(state: MemoryAgentState) -> dict[str, Any]:
    save_messages(
        state["messages"],
        session_id=state["session_id"],
        user_id=state["user_id"],
        source="langgraph_tool_agent",
    )
    return {}


agent_builder = StateGraph(MemoryAgentState)
agent_builder.add_node("call_model", call_model)
agent_builder.add_node("tool_node", tool_node)
agent_builder.add_node("save_agent_log", save_agent_log)

agent_builder.add_edge(START, "call_model")
agent_builder.add_conditional_edges("call_model", should_continue, ["tool_node", "save_agent_log"])
agent_builder.add_edge("tool_node", "call_model")
agent_builder.add_edge("save_agent_log", END)

memory_agent = agent_builder.compile()
memory_agent


In [ ]:
agent_user_message = HumanMessage(
    content=(
        "Loop demo: call one tool per turn. First save that I prefer bullet-point "
        "implementation plans. Then get the current datetime for Asia/Bangkok. "
        "Then look up my user profile, search chat history for MongoDB, save a short "
        "agent note titled 'memory demo follow-up', list recent agent notes, get memory "
        "stats, and finally summarize what you learned."
    )
)


def short_json(value: Any, limit: int = 1200) -> str:
    text = json.dumps(json_safe(value), ensure_ascii=False, indent=2, default=str)
    if len(text) > limit:
        return text[:limit] + "\n...<truncated>"
    return text


def short_text(value: Any, limit: int = 1200) -> str:
    if isinstance(value, list):
        value = json.dumps(value, ensure_ascii=False, default=str)
    text = str(value)
    if len(text) > limit:
        return text[:limit] + "\n...<truncated>"
    return text


def stream_print_text(value: Any, *, chunk_size: int = 16, delay: float = 0.01, limit: int = 4000) -> None:
    """Print a completed LLM response in visible streamed chunks inside the turn trace."""
    import time

    text = short_text(value, limit=limit)
    for start in range(0, len(text), chunk_size):
        print(text[start : start + chunk_size], end="", flush=True)
        if delay:
            time.sleep(delay)
    print()


def print_agent_turn(turn_number: int, node_name: str, messages: list[BaseMessage]) -> None:
    print(f"\nTurn {turn_number} | {node_name}")
    print("=" * 72)

    if node_name == "call_model":
        for message in messages:
            tool_calls = getattr(message, "tool_calls", None) or []
            if tool_calls:
                for index, tool_call in enumerate(tool_calls, start=1):
                    print(f"LLM call tool #{index}: {tool_call['name']}")
                    print("Arguments:")
                    print(short_json(tool_call.get("args", {})))
            else:
                print("LLM final text response streamed:")
                stream_print_text(message.content)

    elif node_name == "tool_node":
        for message in messages:
            print(f"Tool response: {getattr(message, 'name', 'unknown_tool')}")
            print(f"Tool call id: {getattr(message, 'tool_call_id', '')}")
            print("Observation:")
            print(short_text(message.content))

    elif node_name == "save_agent_log":
        print("Conversation turn saved to MongoDB history.")

    else:
        for message in messages:
            print(short_text(getattr(message, "content", message)))


agent_input = {
    "messages": [agent_user_message],
    "session_id": SESSION_ID,
    "user_id": USER_ID,
    "llm_calls": 0,
}

print("Turn 0 | user")
print("=" * 72)
print(agent_user_message.content)

LLM_TOKEN_USAGE_LOG.clear()
trace_messages = [agent_user_message]
llm_call_count = 0
turn_number = 1

for event in memory_agent.stream(
    agent_input,
    config={"recursion_limit": 20},
    stream_mode="updates",
):
    for node_name, update in event.items():
        update = update or {}
        messages = update.get("messages", [])
        if messages:
            trace_messages.extend(messages)
        if "llm_calls" in update:
            llm_call_count = update["llm_calls"]
        if not (TRACE_STREAM_CALL_MODEL and node_name == "call_model"):
            print_agent_turn(turn_number, node_name, messages)
        turn_number += 1

agent_result = {
    "messages": trace_messages,
    "session_id": SESSION_ID,
    "user_id": USER_ID,
    "llm_calls": llm_call_count,
}


agent_token_usage_total = summarize_token_usage(LLM_TOKEN_USAGE_LOG)
agent_result["token_usage_by_turn"] = LLM_TOKEN_USAGE_LOG.copy()
agent_result["token_usage_total"] = agent_token_usage_total

print("\nToken Usage By LLM Turn")
print("=" * 72)
if LLM_TOKEN_USAGE_LOG:
    token_usage_df = pd.DataFrame(LLM_TOKEN_USAGE_LOG)[[
        "llm_call",
        "turn",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
        "source",
        "backend",
    ]]
    display(token_usage_df)
else:
    print("No LLM token usage recorded.")

print("\nAll Token Usage")
print("=" * 72)
print(f"LLM calls: {agent_token_usage_total['llm_calls']:,}")
print(f"Prompt tokens: {agent_token_usage_total['prompt_tokens']:,}")
print(f"Completion tokens: {agent_token_usage_total['completion_tokens']:,}")
print(f"Total tokens: {agent_token_usage_total['total_tokens']:,}")


## 11. Inspect MongoDB Memory

After running the agent, inspect both the append-only history collection and the durable profile collection.


In [ ]:
print("Durable user profile")
pprint(profile_collection.find_one({"user_id": USER_ID}, {"_id": 0}))

print("\nRecent history")
pd.DataFrame(load_recent_history(limit=12))[["created_at", "role", "source", "content"]]


## 12. Summarize the Complete Context

A production app often compacts context before sending it to the model. This summary combines the main ingredients of an LLM call: system prompt, user message, tools, and history.


In [ ]:
def summarize_context(
    *,
    system_prompt: str,
    user_message: str,
    tools: list[Any],
    history_docs: list[dict[str, Any]],
) -> str:
    tool_text = "\n".join(
        f"- {memory_tool.name}: {memory_tool.description}"
        for memory_tool in tools
    )
    history_text = render_history(history_docs)

    summary_prompt = f"""
Summarize the context for an AI application call.

Return concise Markdown with these sections:
1. System prompt intent
2. User message
3. Available tools
4. Relevant history
5. Memory updates or risks

System prompt:
{system_prompt}

User message:
{user_message}

Tools:
{tool_text}

History:
{history_text}
""".strip()
    response = stream_llm_text([
        SystemMessage(content="You summarize LLM runtime context for developers."),
        HumanMessage(content=summary_prompt),
    ])
    
    return response


context_summary = summarize_context(
    system_prompt=TOOL_AGENT_SYSTEM_PROMPT.format(
        user_id=USER_ID,
        session_id=SESSION_ID,
        history=render_history(load_recent_history(limit=8)),
    ),
    user_message=agent_user_message,
    tools=tools,
    history_docs=load_recent_history(limit=12),
)

summary_collection.insert_one(
    {
        "session_id": SESSION_ID,
        "user_id": USER_ID,
        "summary": context_summary,
        "created_at": utc_now(),
    }
)

print(context_summary)


## 13. Memory Design Notes

- Keep raw history and durable profile memory separate. History is an event log; profile memory is a structured record.
- Add retention policy and deletion flows before production use.
- Avoid storing secrets or sensitive personal data by default.
- Use indexes on `session_id`, `user_id`, and `created_at` for common memory retrieval paths.
- In production, replace substring search with vector search, full-text search, or a hybrid retriever when recall matters.
- Summaries are useful for context compression, but keep links back to source history so the app can audit what was summarized.


## 14. One Command: Chat With Agent and Save Token Usage to MongoDB

This section wraps the full agent chat flow into one callable command. It streams the agent loop, prints tool calls and tool responses by turn, summarizes token usage per LLM turn and across the whole run, and saves the token usage record to MongoDB.

Run one command:

```python
one_command_result = chat_with_memory_agent("Your message here")
```


In [ ]:
TOKEN_USAGE_COLLECTION = os.getenv("MONGODB_TOKEN_USAGE_COLLECTION", "agent_token_usage")
token_usage_collection = db[TOKEN_USAGE_COLLECTION]
token_usage_collection.create_index("run_id", unique=True)
token_usage_collection.create_index([("session_id", ASCENDING), ("created_at", ASCENDING)])
token_usage_collection.create_index([("user_id", ASCENDING), ("created_at", ASCENDING)])


def extract_final_assistant_text(messages: list[BaseMessage]) -> str:
    for message in reversed(messages):
        if isinstance(message, AIMessage) and not (getattr(message, "tool_calls", None) or []):
            return short_text(message.content, limit=4000)
    return ""


def extract_tool_sequence(messages: list[BaseMessage]) -> list[dict[str, Any]]:
    sequence = []
    for message in messages:
        tool_calls = getattr(message, "tool_calls", None) or []
        for tool_call in tool_calls:
            sequence.append(
                {
                    "event": "llm_tool_call",
                    "tool_call_id": tool_call.get("id"),
                    "tool_name": tool_call.get("name"),
                    "args": json_safe(tool_call.get("args", {})),
                }
            )
        if isinstance(message, ToolMessage):
            sequence.append(
                {
                    "event": "tool_response",
                    "tool_call_id": getattr(message, "tool_call_id", None),
                    "tool_name": getattr(message, "name", None),
                    "observation": short_text(message.content, limit=2000),
                }
            )
    return sequence


def save_agent_token_usage_to_mongodb(
    *,
    run_id: str,
    session_id: str,
    user_id: str,
    user_message: str,
    result: dict[str, Any],
) -> str:
    doc = {
        "run_id": run_id,
        "session_id": session_id,
        "user_id": user_id,
        "user_message": user_message,
        "backend": LLM_BACKEND_ACTIVE,
        "llm_calls": result.get("llm_calls", 0),
        "token_usage_by_turn": json_safe(result.get("token_usage_by_turn", [])),
        "token_usage_total": json_safe(result.get("token_usage_total", {})),
        "tool_sequence": json_safe(extract_tool_sequence(result.get("messages", []))),
        "final_assistant_text": extract_final_assistant_text(result.get("messages", [])),
        "created_at": utc_now(),
    }
    insert_result = token_usage_collection.insert_one(doc)
    return str(insert_result.inserted_id)


def print_token_usage_summary(usage_log: list[dict[str, Any]], totals: dict[str, Any]) -> None:
    print("\nToken Usage By LLM Turn")
    print("=" * 72)
    if usage_log:
        token_usage_df = pd.DataFrame(usage_log)[[
            "llm_call",
            "turn",
            "prompt_tokens",
            "completion_tokens",
            "total_tokens",
            "source",
            "backend",
        ]]
        display(token_usage_df)
    else:
        print("No LLM token usage recorded.")

    print("\nAll Token Usage")
    print("=" * 72)
    print(f"LLM calls: {totals['llm_calls']:,}")
    print(f"Prompt tokens: {totals['prompt_tokens']:,}")
    print(f"Completion tokens: {totals['completion_tokens']:,}")
    print(f"Total tokens: {totals['total_tokens']:,}")


def chat_with_memory_agent(
    user_text: str,
    *,
    session_id: str = SESSION_ID,
    user_id: str = USER_ID,
    recursion_limit: int = 20,
    save_token_usage: bool = True,
) -> dict[str, Any]:
    """One command to chat with the memory agent and persist token usage to MongoDB."""
    run_id = str(uuid.uuid4())
    user_message = HumanMessage(content=user_text)
    agent_input = {
        "messages": [user_message],
        "session_id": session_id,
        "user_id": user_id,
        "llm_calls": 0,
    }

    print(f"Agent run id: {run_id}")
    print("Turn 0 | user")
    print("=" * 72)
    print(user_text)

    LLM_TOKEN_USAGE_LOG.clear()
    trace_messages = [user_message]
    llm_call_count = 0
    turn_number = 1

    for event in memory_agent.stream(
        agent_input,
        config={"recursion_limit": recursion_limit},
        stream_mode="updates",
    ):
        for node_name, update in event.items():
            update = update or {}
            messages = update.get("messages", [])
            if messages:
                trace_messages.extend(messages)
            if "llm_calls" in update:
                llm_call_count = update["llm_calls"]
            if not (TRACE_STREAM_CALL_MODEL and node_name == "call_model"):
                print_agent_turn(turn_number, node_name, messages)
            turn_number += 1

    token_usage_by_turn = LLM_TOKEN_USAGE_LOG.copy()
    token_usage_total = summarize_token_usage(token_usage_by_turn)
    result = {
        "run_id": run_id,
        "messages": trace_messages,
        "session_id": session_id,
        "user_id": user_id,
        "llm_calls": llm_call_count,
        "token_usage_by_turn": token_usage_by_turn,
        "token_usage_total": token_usage_total,
    }

    print_token_usage_summary(token_usage_by_turn, token_usage_total)

    if save_token_usage:
        mongo_id = save_agent_token_usage_to_mongodb(
            run_id=run_id,
            session_id=session_id,
            user_id=user_id,
            user_message=user_text,
            result=result,
        )
        result["token_usage_mongodb_id"] = mongo_id
        print("\nSaved token usage to MongoDB")
        print("=" * 72)
        print(f"Collection: {TOKEN_USAGE_COLLECTION}")
        print(f"run_id: {run_id}")
        print(f"mongodb_id: {mongo_id}")

    return result


In [ ]:
answer = chat_with_memory_agent("Hello My name is Ro. Nice too meet you.")

In [ ]:
answer["messages"][-1].content

In [ ]:
while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ("exit", "quit"):
        print("Exiting chat.")
        break
    answer = chat_with_memory_agent(user_input)
    print(f"\nAssistant: {answer['messages'][-1].content}")